python scripts/gello_get_offset.py \
    --start-joints 0 -1.57 1.57 -1.57 -1.57 0 \
    --joint-signs 1 1 -1 1 1 1 \
    --port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0


python scripts/gello_get_offset.py \
    --start-joints -1.57 -1.57 -1.57 -1.57 1.57 0 \
    --joint-signs 1 1 -1 1 1 1 \
    --port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0




ls /dev/serial/by-id/


sudo chmod 666 /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0

临时授权（无需重启，立刻能用）
既然你不想重启电脑，我们可以用管理员权限直接把这个 USB 端口的“门”打开，允许所有人访问。

（注意：这只是针对这次插拔有效。如果你拔掉 USB 再插上，或者重启电脑，可能需要再输一次。）

运行完上面这句后，立刻再次运行你的校准脚本，应该就能成功连接了。


ls /dev/video* 获得 camera_index


sudo usermod -aG dialout $USER

什么是 dialout 用户组？为什么要加？
这是一个**权限（Permission）**问题。

通俗解释：

现状：在 Linux 系统眼里，USB 串口（用来控制舵机的那根线）是受保护的“重要硬件”。默认情况下，只有 root（超级管理员）才有资格读写它。你当前登录的普通用户 brian 是没有资格去碰它的。如果你直接运行脚本，系统会报错：Permission denied（拒绝访问）。

dialout 是什么：Linux 系统里有一个特定的用户组叫 dialout。凡是加入这个组的用户，就被系统赋予了“可以随意操作串口设备”的特权。

操作目的：我们要把你的账户 brian 拉进这个 dialout 微信群，这样你以后操作舵机就不用每次都输密码或者是被拒绝了。

python experiments/launch_nodes.py --robot sim_ur

python experiments/launch_nodes.py --robot ur

python experiments/run_env.py --agent=gello --use_save_interface

python -m visual_guided_collection_gui.main \
  --operation-mode demo \
  --wrist-camera Orbbec \
  --gello-port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0 \
  --disable-ultrasound

python -m visual_guided_collection_gui.main \
  --operation-mode demo \
  --control-tcp \
  --wrist-camera Orbbec \
  --gello-port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0 \
  --force-ip 192.168.1.100 \
  --disable-ultrasound

python -m visual_guided_collection_gui.main \
  --operation-mode auto \
  --control-tcp \
  --auto-scan-safe-retreat \
  --wrist-camera Orbbec \
  --gello-port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0 \
  --force-ip 192.168.1.100 \
  --disable-ultrasound \
  --skip-rgb-depth-recording

python -m visual_guided_collection_gui.main \
  --operation-mode comparison \
  --wrist-camera Orbbec \
  --gello-port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0 \
  --force-ip 192.168.1.100 \
  --disable-ultrasound
  --skip-rgb-depth-recording

python -m visual_guided_collection_gui.main \
  --operation-mode bo \
  --wrist-camera Orbbec \
  --gello-port /dev/serial/by-id/usb-FTDI_USB__-__Serial_Converter_FTAW77E7-if00-port0 \
  --force-ip 192.168.1.100

即便你的 UR5 没有安装末端夹爪（爪子），你依然可以使用 GELLO 进行控制。GELLO 的核心原理是“关节角度映射”，你只需要把 GELLO 手柄上前 6 个关节的角度映射给 UR5 的 6 个关节，忽略掉第 7 个关节（通常用于控制夹爪）即可。

以下是针对“**无爪 UR5 + GELLO**”的具体配置步骤：

### 第一步：安装必要依赖 (Python 环境)

除了 GELLO 自身的依赖，UR 机器人需要专门的通信库 `ur_rtde`。

1. **进入你的虚拟环境** (如果你之前建了的话)：
```bash
source .venv/bin/activate

```


2. **安装 UR 通信库**：
```bash
pip install ur_rtde

```



### 第二步：硬件连接与网络设置

1. **GELLO 连接**：
* 将 GELLO 的 USB (U2D2) 插入电脑。
* 确认端口号，通常是 `/dev/ttyUSB0` 或者 `/dev/serial/by-id/xxxx`。


2. **UR5 连接**：
* 用网线将电脑和 UR5 控制箱直连。
* **设置静态 IP**：
* 在 UR5 的示教器上查看网络设置（例如：`192.168.1.102`）。
* 把你的电脑有线网卡 IP 设置为同网段（例如：`192.168.1.100`）。
* **测试**：在终端 `ping 192.168.1.102` 确保能通。





### 第三步：修改配置文件（关键步骤）

由于官方文档主要针对带夹爪的 YAM 机器人，你需要手动修改代码来适配“无爪 UR5”。

**1. 修改机器人 IP 地址**
打开文件 `gello/robots/ur.py`：

* 找到 `__init__` 函数或 IP 设置的地方。
* 将默认 IP 修改为你 UR5 的实际 IP（比如 `192.168.1.102`）。
* *注意：如果该文件引用了外部 config，请确保在对应位置修改。*

**2. 配置 GELLO Agent (屏蔽夹爪)**
打开文件 `gello/agents/gello_agent.py`，你需要告诉程序“我的手柄只读前 6 个电机”。

找到 `PORT_CONFIG_MAP` 字典，根据你的 USB 端口添加或修改配置：

```python
# gello/agents/gello_agent.py

# ... (在 PORT_CONFIG_MAP 中)
"/dev/serial/by-id/你的_USB_设备_ID": AgentConfig(
    # UR5 需要 6 个关节
    dynamixel_config=DynamixelRobotConfig(
        # 【重点】这里只写 1 到 6，不要写 7 (或者是 gripper 的那个 ID)
        # 这样 GELLO 就只会读取前 6 个关节，完全忽略夹爪电机
        joint_ids=(1, 2, 3, 4, 5, 6), 
        joint_offsets=(0, 0, 0, 0, 0, 0),  # 稍后通过校准脚本填入
        joint_signs=(1, 1, -1, 1, 1, 1),   # UR 的推荐方向，参考 README
        gripper_config=None  # 【重点】设置为 None，表示没有夹爪
    ),
    robot_type=URRobot,  # 指定控制对象是 UR 机器人
),

```

*这样做的好处是：软件层面上完全切断了夹爪的数据流，防止程序因为找不到夹爪或者数据维度不匹配（发了 7 个数据给只接受 6 个数据的 UR5）而报错。*

### 第四步：校准零点 (Calibration)

你需要让 GELLO 手柄的姿态和 UR5 的初始姿态保持一致，以计算偏移量。

1. **摆姿势**：
* 将 UR5 移动到一个标准的初始位置（例如所有关节归零，或者竖直向上）。
* 手动将 GELLO 手柄摆成**一模一样**的姿势。


2. **运行校准脚本**：
```bash
python scripts/gello_get_offset.py \
    --start-joints 0 -1.57 1.57 -1.57 -1.57 0 \
    --joint-signs 1 1 -1 1 1 1 \
    --port /dev/serial/by-id/你的_USB_ID

```


*(注：`--start-joints` 后面的数字应为你 UR5 当前实际的关节角度，单位是弧度)*
3. **填回配置**：
* 脚本运行后会吐出一串 `joint_offsets` 数字。
* 把这串数字填回到 `gello/agents/gello_agent.py` 里对应的 `joint_offsets` 字段中。



### 第五步：启动

因为你是非 YAM 机器人，需要用**双终端**方式启动。

**终端 1：启动机器人驱动节点**

```bash
# 这会连接 UR5 并等待指令
python experiments/launch_nodes.py --robot ur

```

*如果报错连接失败，请检查 `gello/robots/ur.py` 里的 IP 是否改对了。*

**终端 2：启动 GELLO 控制器**

```bash
# 这会连接 GELLO 手柄并开始发送指令
python experiments/run_env.py --agent=gello

```

### 总结

对于“无爪”的情况，核心 trick 就是在 **`gello_agent.py`** 中配置 `joint_ids` 时**只保留前 6 个电机 ID**。这样 GELLO 就变成了一个纯粹的 6 自由度输入设备，完美匹配 UR5 的 6 自由度，程序就不会去管那个不存在的爪子了。

这是一个非常好的想法！使用仿真器（Simulator）是调试 GELLO 最安全、最高效的方法。在这一步你不需要连接真实的 UR5 机械臂，只需要连接你的 GELLO 手柄即可。

这个项目使用的是 **MuJoCo** 物理引擎来模拟 UR5。

以下是具体操作步骤：

### 第一步：确认依赖安装

确保你还在那个虚拟环境里（如果没有，记得 `source .venv/bin/activate`）。
仿真需要安装 MuJoCo 相关的库，请运行：

```bash
uv pip install mujoco dm_control

```

*(通常 `requirements.txt` 里已经包含了，但为了保险起见确认一下)*

---

### 第二步：查看仿真器中的“标准姿势”

在校准 GELLO 之前，你必须知道仿真里的 UR5 默认是长什么样子的，这样你才能把 GELLO 摆成一样的姿势。

运行这个命令启动仿真环境：

```bash
python experiments/launch_nodes.py --robot sim_ur

```

* **预期结果：** 会弹出一个窗口，里面有一个悬浮的 UR5 机械臂。
* **观察：** 仔细观察它的姿势（通常是所有关节归零，或者像图片里那样呈“门”字形）。
* **操作：** 观察完后，按 `Ctrl+C` 关闭终端，结束程序。

---

### 第三步：校准 GELLO 手柄

就像控制真机一样，你需要告诉电脑：“当我的 GELLO 摆成这个姿势时，它对应的是仿真里的初始姿势”。

1. **摆姿势：** 用手把 GELLO 手柄摆成你刚才在仿真器里看到的那个姿势。
2. **连接 USB：** 确保 GELLO 的 USB 线插在电脑上。
3. **运行校准脚本：** (假设你是 UR 机器人，Joint Signs 按照官方文档填)

```bash
python scripts/gello_get_offset.py \
    --start-joints 0 -1.57 1.57 -1.57 -1.57 0 \
    --joint-signs 1 1 -1 1 1 1 \
    --port /dev/serial/by-id/你的_USB_ID

```

*注意：`--start-joints` 后面的数字是 UR5 在仿真里的初始角度（弧度制）。如果你刚才看到的仿真机器人是直立的，可能全是 0；如果是弯曲的（如 `-1.57` 即 -90度），请参考 README 中 UR Robot 的示例参数。*

4. **保存数据：** 脚本会输出一串 `joint_offsets`。把这串数字复制下来，去修改 `gello/agents/gello_agent.py` 文件（找到你的 USB 端口那一行，替换进去）。

---

### 第四步：正式启动仿真控制

现在一切准备就绪，我们需要打开**两个终端窗口**。

#### 终端 1：启动“虚拟机器人” (服务端)

这个窗口负责运行物理引擎，模拟机器人的运动。

```bash
python experiments/launch_nodes.py --robot sim_ur

```

* 你应该能看到仿真窗口弹出来。

#### 终端 2：启动“GELLO 控制器” (客户端)

这个窗口负责读取你手上的硬件数据，并发给仿真器。

```bash
python experiments/run_env.py --agent=gello

```

* **预期结果：**
* 终端里会显示 Connection successful 之类的提示。
* **现在，移动你手里的 GELLO 手柄，屏幕上的虚拟 UR5 应该会跟着动！**



---

### 常见问题 (Troubleshooting)

**Q: 屏幕上的机器人动了，但是动作很鬼畜/方向是反的？**

* **A:** 这是 `joint_signs` (关节方向) 没设置对。
* **解决：** 打开 `gello/agents/gello_agent.py`，找到 `joint_signs=(1, 1, -1, ...)` 那一行。试着把那个动作反了的关节的 `1` 改成 `-1`（或者反过来），保存文件，重启 **终端 2** 即可。

**Q: 仿真里的机器人有夹爪，但我不想控制夹爪？**

* **A:** 没关系。仿真模型自带夹爪是为了通用性。只要你的手臂动作（前 6 个关节）映射是对的，那个夹爪你可以当它不存在，或者它会随着你手柄上的扳机空动，不影响手臂轨迹。

**Q: 报错 `ModuleNotFoundError`？**

* **A:** 忘记进入虚拟环境了。记得每个新开的终端都要运行 `source .venv/bin/activate`。

这是一个非常硬核且高质量的问题。你正在触及机器人遥操作（Teleoperation）中最核心的数学原理：**“运动学映射”与“零点标定”**。

简单回答你的核心疑问：

1. **代码在做什么？** 它在计算**“物理安装误差”**。你的电机安装角度是随机的（比如拧螺丝时歪了90度），代码需要算出一个“补偿值（Offset）”，让电机的 0 度对应上机器人的 0 度。
2. **手臂要放得很标准吗？**
* **标定代码 (`gello_get_offset.py`) 运行时：** **不用非常标准，详见后面！** 。
* **仿真/控制代码 (`run_env.py`) 启动时：** **必须比较标准。** 启动瞬间，系统会检查你的手柄是否处于安全位置。如果歪得太厉害，系统怕一连接机器人就“飞”出去，会触发安全保护并退出（就是你刚才遇到的 Timeout/退出）。连接成功后，就可以随意动了。



---

### 详细代码与参数解析

我们把你提供的代码拆解开，看看数学上到底发生了什么。

#### 1. 标定脚本 `gello_get_offset.py`

这个脚本的任务是解一个方程。

**核心公式：**

$$\text{机器人角度} = \text{符号} \times (\text{电机原始读数} - \text{Offset})$$

我们现在已知：

* **机器人角度**：由 `--start-joints` 告诉电脑（比如 -1.57）。
* **符号**：由 `--joint-signs` 告诉电脑（比如 1 或 -1）。
* **电机原始读数**：脚本通过 USB 从硬件读出来的。
* **未知数**：`Offset`。

脚本通过“暴力搜索（Brute Force）”的方法，在 \pm 8\pi 的范围内，每隔 90度（\pi/2）试一次，看哪个 Offset 能让上面的等式成立。

**重要参数详解：**

* **`--start-joints 0 -1.57 1.57 -1.57 -1.57 0`**
* **意义**：这是 UR5 机器人的**“目标姿态”**。
* **物理含义**：
* `0`：底座正对前方。
* `-1.57` (-\pi/2)：大臂竖直向上（或者水平，取决于UR5定义）。
* `1.57` (\pi/2)：肘部弯曲 90 度。
* **注意**：这通常对应 UR5 经典的**“门字形”**姿态（看起来像个门框）。


* **你的操作**：**你必须把 GELLO 手柄摆成完全一样的“门字形”。**


* **`--joint-signs 1 1 -1 1 1 1`**
* **意义**：**“方向修正”**。
* **物理含义**：
* `1`：你的手柄往左转，机器人也往左转（方向一致）。
* `-1`：你的手柄往左转，电机的读数变大，但机器人那边定义往左是变小。所以需要乘个 -1 翻转过来。


* **来源**：这是由 GELLO 的 3D 打印结构决定的。齿轮啮合可能会导致方向反转。


* **输出结果 `best offsets**`
* **你的数据**：`['1.571', '4.712', ...]`
* **意义**：这就是算出来的补偿值。
* 例如第一个关节 Offset 是 `1.571` (\pi/2)。这意味着你把电机安装在手柄上时，相对于机器人的零点，你**物理上**顺时针多转了 90 度安装。代码以后每次读数都要减去这个 90 度。



#### 2. 配置文件 `gello_agent.py`

这部分是你把算出来的数字“固化”到代码里，给运行程序看。

* **`DynamixelRobotConfig` 类**：
这是定义一个机器人配置的模板。
* **`joint_offsets` (最关键)**：
* **代码**：`1 * np.pi / 2, 3 * np.pi / 2, ...`
* **解析**：这里填的就是标定脚本算出来的 `best offsets function of pi`。
* **作用**：程序启动时，会立刻用这个公式计算当前角度：
`当前角度 = (电机读数 - joint_offsets) * joint_signs`
* 如果这里填错了（比如全是 0），计算出的角度就是错的，导致程序认为你现在的姿势和 `--start-joints` 差得很远。


* **`joint_signs`**：
* 直接照抄命令行的配置。确保方向一致。


* **`gripper_config=(7, 196, 154)`** (即使你不用夹爪)
* **7**：夹爪电机的 ID。
* **196 / 154**：这通常是**原始电机位置读数**（单位可能是度或者原始单位，看驱动实现）。
* **196**：表示松手时（Open），电机应该转到 196 的位置。
* **154**：表示握紧时（Closed），电机应该转到 154 的位置。
* **作用**：当你按下手柄扳机，程序会把这个 0~1 的模拟量映射到 154~196 之间发给电机。



---

### 总结：你的操作流程 Checklist

为了保证不再出现 Timeout 或者机器人乱动，请严格遵守以下流程：

**阶段一：标定 (运行 `gello_get_offset.py`)**

1. **摆姿势**：看着仿真里的 UR5（或者是图片里的标准姿态），把手里的 GELLO **极度精确地**摆成一模一样的姿势（关节角度目测要准）。
2. **保持不动**：一只手扶稳 GELLO，另一只手敲回车运行脚本。
3. **填数据**：把输出的 Offset 填入 `gello_agent.py`。

**阶段二：启动仿真 (运行 `run_env.py`)**

1. **归位**：在运行命令前，**先把 GELLO 摆回刚才标定时的那个“门字形”姿势**。
2. **启动**：运行命令。
3. **连接**：程序启动瞬间会计算 `Delta`（当前计算角度 - 预设起始角度）。
* 因为你摆回了原位，且 Offsets 填对了，所以 `Delta` 应该接近 0。
* 程序判定“安全”，建立连接。


4. **开始操作**：连接建立后，你就可以随意动了。

**一句话总结**：
标定就是为了算出你安装时的“歪度”；启动时必须摆正，是为了让系统确认“标定数据是正确的”且“现在连接是安全的”。










你的观察**非常敏锐且专业**！你抓住了这段代码的核心逻辑：**它是离散的“四舍五入”标定。**

确实，我之前的表述“歪 5 度，以后就歪 5 度”在很多连续标定的机器人系统（如工业机械臂、高精度伺服）中是成立的，但在 **GELLO 的这个特定脚本**里，你是对的——**它有一定的“容错空间”。**

---

### 1. 为什么它是离散的穷举？

看这段代码：

```python
for offset in np.linspace(-8 * np.pi, 8 * np.pi, 8 * 4 + 1):  # intervals of pi/2

```

这里 `8*4 + 1 = 33` 个点，在 -8\pi 到 8\pi 之间平分，步长正好是 **\pi/2 (即 90 度)**。

**原理如下：**
GELLO 的硬件设计（舵机盘、3D 打印件、齿轮）通常是按 90 度步进安装的。代码假设你的安装误差**只可能是 90 度的倍数**（比如你把舵机装反了、侧着装了）。

* **如果你摆歪了 5 度**：由于 5 度离 0 度比离 90 度近，代码通过穷举发现 `offset = 0` 时的误差最小，最终它会**强行选择 0** 作为 Offset。
* **结果**：这 5 度的误差被“过滤”掉了，最终运行代码时，系统依然认为你的零点是完美的。

### 2. 那么“必须非常标准”的真正含义是什么？

虽然代码有容错，但这个“标准”依然重要，原因有两个：

#### A. 别掉进“错误的坑”里

离散标定的容错范围是 **\pm 45 度**。

* 如果你摆歪了 5 度，它会修正到 0 度（正确）。
* 如果你摆歪了 50 度，它会认为你物理安装时就是歪着 90 度装的，它会把 Offset 设为 90 度。**这时候你的系统就彻底乱了。**

#### B. 夹爪（Gripper）是连续的！

注意脚本最后关于夹爪的部分：

```python
print("gripper open (degrees)", np.rad2deg(driver.get_joints()[-1]) - 0.2)

```

**夹爪的标定是连续的（Floating point）**。如果你在标定时，手指捏着的力度不一样，或者扳机位置差了一点，这个 `195.53` 或 `153.73` 的数值就会变。这直接影响你遥操作时捏东西的松紧度。

---

### 3. 重要参数深度解析

基于你提供的两段核心代码，这里是它们在数学和物理上的终极对应关系：

#### `joint_offsets` (关节偏移)

* **物理意义**：解决“**安装随机性**”。
* **标定逻辑**：它是电机内部编码器的原始零点与机器人建模零点之间的差值。因为 GELLO 结构允许你在 0/90/180/270 度位置锁死舵机，所以代码用了 `k * pi/2` 的离散值。
* **代码体现**：`1 * np.pi / 2` 这种写法就是为了让人一眼看出“哦，这个关节是侧着装的”。

#### `joint_signs` (关节极性)

* **物理意义**：解决“**镜像问题**”。
* **为什么有 -1**：
* **连杆方向**：UR5 的某些关节（如 Elbow）你往前推，电机顺时针转；但另一个关节你往前推，电机可能逆时针转。
* **安装正反**：舵机正着贴在左边和反着贴在右边，读数增减是相反的。


* **重要性**：如果这个参数错了，你会发现你手柄往左，机器人往右，根本无法操作。

#### `gripper_config` (夹爪配置)

* **参数：`(ID, Open_Pos, Closed_Pos)**`
* **`7`**：这是第 7 个电机的 ID（夹爪）。
* **`196, 154`**：这是电机的**原始电位器读数**。
* **映射关系**：当你操作时，程序会做一个 `lerp`（线性插值）：
* 你手柄扳机开度 0% \rightarrow 发送指令 196 给电机。
* 你手柄扳机开度 100% \rightarrow 发送指令 154 给电机。


* **注意**：如果这两个数字写反了，你的夹爪就会变成“按下去张开，松开握紧”。

#### `start_joints` (初始姿态)

* **物理意义**：这是你的**“锚点”**。
* **作用**：在 `run_env.py` 启动时，它会不断对比：`当前计算出的角度` vs `start_joints`。
* **报错触发**：当你看到的 `joint[x]: delta: 1.504`，就是因为这个差值超过了阈值（通常是 0.5 弧度）。

---

### 4. 总结：你现在的操作策略

既然你发现了它是离散穷举，你可以稍微“偷懒”一点：

1. **手臂关节**：只要目测摆得**大体上**像那个“门字形”（误差在 20 度以内），标定出的 `joint_offsets` 就会是完全正确的。
2. **夹爪**：如果你在乎夹爪的控制精度，标定时请务必确保扳机处于完全松开的状态。
3. **填代码**：直接把你跑出来的 `best offsets function of pi` 那一行完整地覆盖到 `gello_agent.py` 里。

**最后确认一个细节**：
你在 `gello_agent.py` 里定义的 `joint_offsets` 现在有 7 个值还是 6 个值？

* 如果你的 UR5 **没有爪子**，请确保 `joint_ids` 是 `(1, 2, 3, 4, 5, 6)`，`joint_offsets` 也是 6 个值。
* 我看你刚才发的配置里，有的地方写了 7 个值（带夹爪），有的地方写了 6 个值。**一定要保持维度匹配。**

**下一步建议：**
按照我给你的那个 `PORT_CONFIG_MAP` 修改代码，然后尝试再次运行 `run_env.py`。如果 `delta` 变小了，你就成功了！
想让我帮你检查一下你最终修改后的 `PORT_CONFIG_MAP` 字典代码吗？